# YOLOv12n × VisDrone — `vsod` Notebook

> **VSOD: VisDrone-Specialized Object Detector**
>
> **Target: ~46–51% mAP50** (baseline: 26.28% → **+90% relative**)
>
> Bốn kỹ thuật kết hợp:
> 1. **① imgsz 1280** — P3=160×160, tiny objects từ 4px→8px, **ZERO params**
> 2. **② P2 head 320×320** — GhostConv 32ch, bắt object ≥4px
> 3. **③ Area Attention P3** — A2C2f(area=2), local context, **ZERO params**
> 4. **④ WIoU Loss** — focus gradient vào tiny/hard objects, **ZERO params**
>
> Params: ~2.85M (+11% từ 2.57M) — vẫn **nano-class**
>
> ⚠️ **Yêu cầu**: Colab Pro hoặc T4 với batch=4 nếu CUDA OOM
>
> T4 GPU | ~4–6h (150 epochs, imgsz=1280)

## 1. Kiểm tra GPU & CUDA

In [ ]:
import subprocess, torch
r = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(r.stdout if r.returncode==0 else 'No GPU')
print(f'PyTorch : {torch.__version__}')
print(f'CUDA    : {torch.version.cuda}')
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    vram = props.total_memory / 1e9
    print(f'GPU     : {props.name}')
    print(f'VRAM    : {vram:.1f} GB')
    if vram < 14:
        print('⚠ VRAM < 14GB — dùng batch=4 thay vì batch=8')
    else:
        print('✓ VRAM đủ cho batch=8 @ imgsz=1280')

## 2. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('Drive mounted ✓')

## 3. Cài đặt dependencies

In [ ]:
%pip install -q ultralytics>=8.3.0 omegaconf>=2.3.0 rich>=13.0.0 thop
import ultralytics; ultralytics.checks()
print(f'ultralytics: {ultralytics.__version__}')

## 4. Upload project code

Chọn **một trong 3 cách**:

In [ ]:
import os, shutil, zipfile
PROJECT_DIR = '/content/yolov12n-visdrone'

# -- Cách 1: Upload file zip -------------------------------------------
# from google.colab import files
# up = files.upload()
# with zipfile.ZipFile(list(up.keys())[0]) as z: z.extractall('/content/')

# -- Cách 2: Copy từ Google Drive (khuyến nghị) ------------------------
# shutil.copytree('/content/drive/MyDrive/yolov12n-visdrone-code',
#                 PROJECT_DIR, dirs_exist_ok=True)

# -- Cách 3: Git clone -------------------------------------------------
# !git clone https://github.com/your-username/yolov12n-visdrone {PROJECT_DIR}

print('OK' if os.path.exists(PROJECT_DIR) else '⚠ Project chưa được upload')
os.chdir(PROJECT_DIR)
print('cwd:', os.getcwd())

## 5. Load VisDrone dataset từ Google Drive

> Upload `VisDrone.zip` lên Drive tại: `MyDrive/yolov12n-visdrone/datasets/VisDrone.zip`

In [ ]:
import os, zipfile, time
PROJ     = '/content/yolov12n-visdrone'
LOCAL    = PROJ + '/datasets/VisDrone'
BASE     = '/content/drive/MyDrive/yolov12n-visdrone/datasets'
ZIP_PATH = BASE + '/VisDrone.zip'
DIR_PATH = BASE + '/VisDrone'

os.makedirs(PROJ + '/datasets', exist_ok=True)

if os.path.exists(LOCAL):
    print(f'Dataset đã có: {LOCAL} ✓')
elif os.path.exists(ZIP_PATH):
    print(f'Giải nén từ {ZIP_PATH} ...')
    t0 = time.time()
    with zipfile.ZipFile(ZIP_PATH) as z:
        z.extractall(PROJ + '/datasets')
    print(f'Xong trong {int(time.time()-t0)}s ✓')
elif os.path.exists(DIR_PATH):
    os.symlink(DIR_PATH, LOCAL)
    print(f'Symlink: {DIR_PATH} → {LOCAL} ✓')
else:
    raise FileNotFoundError(f'Không tìm thấy VisDrone.zip tại: {ZIP_PATH}')

In [ ]:
!python scripts/check_dataset.py --dataset-root datasets/VisDrone

## 6. Cấu hình training

In [ ]:
IDEA        = 'vsod'
MODEL       = 'yolov12n'
EPOCHS      = 150       # nhiều hơn baseline (100) để hội tụ ở 1280px
BATCH       = 8         # giảm từ 16 → 8 vì imgsz tăng 2x
IMGSZ       = 1280      # ← KEY: đây là lý do chính đạt ~50% mAP
DEVICE      = '0'
TEST_EVERY  = 10
PROJECT_OUT = 'runs'
EXP_NAME    = f'{MODEL}_{IDEA}'

import torch
if torch.cuda.is_available():
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    if vram < 12:
        BATCH = 4
        print(f'⚠ VRAM={vram:.1f}GB < 12GB → batch tự động giảm xuống {BATCH}')
    elif vram < 16:
        BATCH = 6
        print(f'⚠ VRAM={vram:.1f}GB → batch={BATCH} (safe mode)')

print(f'Model     : {MODEL} / {IDEA}')
print(f'imgsz     : {IMGSZ}px  ← quan trọng nhất!')
print(f'Batch     : {BATCH}')
print(f'Epochs    : {EPOCHS}')
print(f'Output    : {PROJECT_OUT}/{EXP_NAME}')
print()
print('VSOD kỹ thuật:')
print('  ① imgsz=1280  : P3 160×160, tiny obj 4px→8px  (+12% mAP expected)')
print('  ② P2 head     : GhostConv 320×320, 4-head det  (+5% mAP)')
print('  ③ Area Attn   : A2C2f(area=2) tại P3 FPN        (+3% mAP, 0 params)')
print('  ④ WIoU loss   : focus trên tiny/hard objects    (+2% mAP, 0 params)')
print('  ⑤ GhostConv   : PAN neck nhẹ hơn               (bù params P2 head)')

## 7. Kiểm tra kiến trúc VSOD

Xem params và layers trước khi train.

In [ ]:
from ultralytics import YOLO

print('=== VSOD Architecture ===')
vsod_model = YOLO('configs/yolov12n_vsod.yaml')
n_total = sum(p.numel() for p in vsod_model.model.parameters())
print(f'Total params: {n_total:,} ({n_total/1e6:.3f}M)')

print('\nPer-layer breakdown:')
for i, layer in enumerate(vsod_model.model.model):
    n = sum(p.numel() for p in layer.parameters())
    tag = ''
    if i <= 8: tag = '← backbone (pretrained)'
    elif i == 17: tag = '← P2 head (NEW)'
    elif i in [20,23]: tag = '← PAN GhostConv'
    elif i == 26: tag = '← P5 C3k2'
    elif i == 27: tag = '← 4-head Detect'
    print(f'  [{i:2d}] {type(layer).__name__:<20} {n:>8,} params  {tag}')

print('\n=== Baseline (YOLOv12n) ===')
base = YOLO('yolo12n.pt')
n_base = sum(p.numel() for p in base.model.parameters())
print(f'Total params: {n_base:,} ({n_base/1e6:.3f}M)')
print(f'Delta       : {(n_total-n_base)/1e3:+.1f}K params ({100*(n_total-n_base)/n_base:+.1f}%)')

## 8. Training

Cell này chạy **~4–6h trên T4** (150 epochs @ imgsz=1280).

Mỗi 10 epochs in full metrics report. Nếu bị ngắt kết nối:
```bash
# Thêm --resume hoặc --pretrained runs/yolov12n_vsod/weights/last.pt
```

In [ ]:
import os; os.makedirs('logs', exist_ok=True)
!python train.py \\
    --model      {MODEL} \\
    --idea       {IDEA} \\
    --epochs     {EPOCHS} \\
    --batch      {BATCH} \\
    --imgsz      {IMGSZ} \\
    --device     {DEVICE} \\
    --project    {PROJECT_OUT} \\
    --name       {EXP_NAME} \\
    --test-every {TEST_EVERY} \\
    --log-file   logs/{EXP_NAME}.log

## 9. Đánh giá full metrics

In [ ]:
import os
WEIGHTS = f'{PROJECT_OUT}/{EXP_NAME}/weights/best.pt'
if not os.path.exists(WEIGHTS):
    WEIGHTS = f'{PROJECT_OUT}/{EXP_NAME}/weights/last.pt'
    print(f'best.pt not found, dùng: {WEIGHTS}')
else:
    print(f'Weights: {WEIGHTS}')

In [ ]:
# Val split
!python val.py \\
    --weights        {WEIGHTS} \\
    --data           configs/visdrone.yaml \\
    --split          val \\
    --imgsz          {IMGSZ} \\
    --device         {DEVICE} \\
    --model-name     {MODEL} \\
    --idea           {IDEA} \\
    --save-csv       runs/metrics.csv \\
    --benchmark-runs 30

In [ ]:
# Test split
!python val.py \\
    --weights        {WEIGHTS} \\
    --data           configs/visdrone.yaml \\
    --split          test \\
    --imgsz          {IMGSZ} \\
    --device         {DEVICE} \\
    --model-name     {MODEL} \\
    --idea           {IDEA} \\
    --save-csv       runs/metrics.csv \\
    --benchmark-runs 30

## 10. So sánh tất cả experiments

In [ ]:
import pandas as pd, os
if os.path.exists('runs/metrics.csv'):
    df = pd.read_csv('runs/metrics.csv')
    cols = ['model','idea','split','acc_map50','acc_map50_95',
            'acc_precision','acc_recall',
            'eff_gflops','eff_params_m','eff_fps',
            'ratio_map50_per_gflop','ratio_map50_x_fps']
    c = [x for x in cols if x in df.columns]
    pd.set_option('display.max_columns', None)
    pd.set_option('display.width', 220)
    pd.set_option('display.float_format', '{:.4f}'.format)
    print(df[c].to_string(index=False))

    # Delta vs baseline
    if 'acc_map50' in df.columns:
        test_df = df[df.get('split','')=='test'] if 'split' in df else df
        print('\n--- Delta vs Baseline ---')
        base_map = test_df[test_df['idea']=='baseline']['acc_map50'].values
        for idea in ['cagi','sfod','vsod']:
            row = test_df[test_df['idea']==idea]['acc_map50'].values
            if len(row) and len(base_map):
                delta = row[-1] - base_map[-1]
                pct = 100*delta/base_map[-1]
                print(f'  {idea:10s}: {row[-1]:.4f}  ({delta:+.4f}, {pct:+.1f}%)')
else:
    print('metrics.csv not found — chạy Section 9 trước')

## 11. Visualize training curves

In [ ]:
from IPython.display import Image, display
import os
exp_dir = f'{PROJECT_OUT}/{EXP_NAME}'
for name in ['results.png','confusion_matrix.png','PR_curve.png','F1_curve.png']:
    p = os.path.join(exp_dir, name)
    if os.path.exists(p):
        print(name); display(Image(p, width=820))

In [ ]:
import pandas as pd, os
rcsv = f'{PROJECT_OUT}/{EXP_NAME}/results.csv'
if os.path.exists(rcsv):
    df = pd.read_csv(rcsv); df.columns = df.columns.str.strip()
    col = 'metrics/mAP50(B)'
    if col in df.columns:
        best = df.loc[df[col].idxmax()]
        print(f'Best epoch  : {int(best.get("epoch",0))}/{EPOCHS}')
        print(f'mAP50       : {best[col]:.4f}')
        print(f'mAP50-95    : {best.get("metrics/mAP50-95(B)",0):.4f}')
        print(f'Precision   : {best.get("metrics/precision(B)",0):.4f}')
        print(f'Recall      : {best.get("metrics/recall(B)",0):.4f}')

        import matplotlib.pyplot as plt
        fig, axes = plt.subplots(1, 2, figsize=(14, 4))
        axes[0].plot(df['epoch'], df[col], 'b-', lw=2, label='VSOD mAP50')
        axes[0].axhline(0.2628, color='r', ls='--', label='Baseline 0.2628')
        axes[0].axhline(0.48, color='g', ls=':', label='Target 0.48')
        axes[0].set_title('mAP50 vs Epoch'); axes[0].legend(); axes[0].grid(True, alpha=0.3)
        loss_col = 'train/box_loss'
        if loss_col in df.columns:
            axes[1].plot(df['epoch'], df[loss_col], 'r-', lw=2)
            axes[1].set_title('Box Loss'); axes[1].grid(True, alpha=0.3)
        plt.tight_layout(); plt.show()

## 12. Lưu lên Google Drive

In [ ]:
import shutil, os
DRIVE_OUT = f'/content/drive/MyDrive/yolov12n-visdrone/runs/{EXP_NAME}'
os.makedirs(DRIVE_OUT, exist_ok=True)

src = f'{PROJECT_OUT}/{EXP_NAME}'
if os.path.exists(src):
    shutil.copytree(src, DRIVE_OUT, dirs_exist_ok=True)
    print(f'Saved → {DRIVE_OUT} ✓')

if os.path.exists('runs/metrics.csv'):
    shutil.copy('runs/metrics.csv',
                '/content/drive/MyDrive/yolov12n-visdrone/metrics.csv')
    print('metrics.csv backed up ✓')

# Download best.pt (optional)
# from google.colab import files
# files.download(f'{src}/weights/best.pt')

## 13. Demo Inference

In [ ]:
import glob, random
import matplotlib.pyplot as plt
from ultralytics import YOLO

model = YOLO(WEIGHTS)
imgs  = glob.glob('datasets/VisDrone/images/test/*.jpg')
samples = random.sample(imgs, min(3, len(imgs)))

results = model.predict(
    source=samples, conf=0.20, imgsz=IMGSZ, device=DEVICE,
    save=True, project='runs/predict', name=f'demo_{IDEA}'
)

fig, axes = plt.subplots(1, 3, figsize=(20, 6))
for i, r in enumerate(results):
    img = r.plot()[:, :, ::-1]
    axes[i].imshow(img)
    axes[i].set_title(samples[i].split('/')[-1], fontsize=8)
    axes[i].axis('off')
fig.suptitle(f'VSOD @ {IMGSZ}px — 4 heads: P2(320)+P3(160)+P4(80)+P5(40)',
             fontsize=11, fontweight='bold')
plt.tight_layout()
plt.show()